## This analysis is concerned with the enrichment of LASV/Mammarenavirus/Arenavirus-specific peptides by LASV monoclonal antibodies. 

In [ ]:
import pandas as pd
df = pd.read_csv('downloads_study920_mapped_rpk.csv').set_index('peptide')
peps_metadata = pd.read_csv('../metadata_files/lassa_library_sequences_species_standardized_070726_ICTV_corrected.csv')

In [7]:
df_mAb = df.filter(regex='mAb')
df_AG = df.filter(regex='AG')

In [12]:
df_AG_avg = df_AG.mean(axis=1)

In [13]:
df_mAb_fc = (df_mAb + 1).div((df_AG_avg + 1), axis=0)

### filter to Arenavirus peptides and glycoprotein

In [40]:
peps_metadata_arenavirus_gp = peps_metadata.loc[(peps_metadata['family'] == 'Arenaviridae') & (peps_metadata['protein_std'] == 'glycoprotein')]
seq_id = peps_metadata_arenavirus_gp['seq_id']

In [41]:
df_mAb_fc_arenaviridae_gp = df_mAb_fc.loc[seq_id]

## now going to investigate each mAb individually

In [144]:
import numpy as np
import pandas as pd
from Bio import Align, SeqIO
from Bio.Align import substitution_matrices 

def analyze_mab_peptides(
    df, 
    peps_metadata, 
    regex, 
    ref_path="../ref_seqs/UP000002473_11622_LASV_Josiah_reference.fasta", 
    target_id="sp|P08669|GLYC_LASSJ", 
    top_n=20, 
    literature_note=None, 
    print_alignments=True
):
    """Select two mAb replicate columns using regex, calculate Pearson's r, only keep if passes 0.95,
    rank peptides by mean enrichment, and align the top peptides to a reference protein. 
    
    Reference coordinates are 1-based and inclusive. 
    Discontinuous covered regions are reported separately. 
    """
    def get_covered_regions(positions):
        """Convert positions into separate contiguous intervals"""
        positions = sorted(set(positions))

        if not positions:
            return[]

        regions = []
        region_start = positions[0]
        region_end = positions[0]

        for position in positions[1:]:

            if position == region_end + 1:
                region_end = position #extend the current region
            else:
                #gap: save the completed region
                regions.append((region_start, region_end))

                #start a new region at the current position
                region_start = position
                region_end = position

        regions.append((region_start, region_end))
        return regions

    def format_regions(regions):
        return ", ".join(
            str(start) if start == end else f"{start}-{end}"
            for start, end in regions
        )

    """calculate pearson's R for Arenavirus GPC peptides and find top 20 Arenavirus GPC peptides"""
    mAbs = df.filter(regex=regex, axis=1).copy()
    
    mAbs = (mAbs.apply(pd.to_numeric, errors='raise'))

    pearsons_r = mAbs.iloc[:, 0].corr(mAbs.iloc[:, 1], method='pearson')

    print(f"Pearson's r = {pearsons_r} for {regex} mAbs")

    if pearsons_r < 0.95: 
        print(f"Correlation too low, Analysis stopped")

        #return exits the function
        return { 
            "pearsons_r": pearsons_r, 
            "passed_correlation": False, 
            "top_peptides": None, 
            "mapping": None,
            "covered_regions": [], 
            "positions_covered": []
        }
        

    ### ---- Rank peptides by mean enrichment 

    mAbs_avg = mAbs.mean(axis=1)

    mAbs_top = mAbs_avg.dropna().sort_values(ascending=False).head(top_n).rename("mean_fc")

    print(mAbs_top)

    ### --- retrieve metadata 
    mAbs_top_pep_sequences = peps_metadata.loc[peps_metadata['seq_id'].isin(mAbs_top.index.to_list())]

    """alignment part""" #---- Load reference

    #SeqIO.to_dict() expects parsed sequence records
    ref_records = SeqIO.to_dict(SeqIO.parse(ref_path, 'fasta'))

    target_id = "sp|P08669|GLYC_LASSJ"
    ref_seq = ref_records[target_id].seq

    #------- configure pairwise aliger for peptides 

    aligner = Align.PairwiseAligner()
    aligner.mode = 'local' #for peptides
    aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -0.5

    #align each peptide to the reference sequence
    peptides = mAbs_top_pep_sequences['sequence'].to_list()

    positions_covered = set()

    for peptide_id, row in mAbs_top_pep_sequences.iterrows():
        pep = str(row['sequence']).strip().upper()

        alignments = aligner.align(ref_seq, pep)
        best_alignment = alignments[0] #top scoring alignment

        if best_alignment is None: 
            continue

        #All aligned blocks; convert to 1-based inclusive positions
        for ref_start, ref_end in best_alignment.aligned[0]:
            positions_covered.update(
                range(int(ref_start) + 1, int(ref_end) +1) #inside range() need ref_end+1, because python excludes the stopping value
            )
        
        if print_alignments:
            print(f"\nPeptide ID: {peptide_id}")
            print(f"Peptide: {pep} | Score: {best_alignment.score}")
            print(best_alignment)


    covered_regions = get_covered_regions(positions_covered)

    print(f"\nResults for {regex}")
    print(f"Pearson's r = {pearsons_r:.3f}")
    print(f"Top peptides selected {len(mAbs_top)}")
    print("Covered reference regions (1-based, inclusive)")


    if literature_note:
        print(literature_note)

    return {
        "pearsons_r": pearsons_r,
        "top_peptides": peptides,
        "peptides enrichment values": mAbs_top,
        "covered_regions": covered_regions,
        "positions_covered": sorted(positions_covered),
    }


    

In [146]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='1-1B', literature_note = "Literature reports LASV 1.1B to bind a linear epitope within the T-loop of the GP2 subunit of GPC, mapped to residues aa361-375, non-neutralizing")

Pearson's r = 0.9970396135453448 for 1-1B mAbs
peptide
QIM73713.1|precursor|Arenavirus_sp.|RtRsp-AreV/Tt2018|Rattus_sp.|Thailand|Nov-2018|?|?|tile_17                                                                         795.207770
ABM46592.1|precursor|Mammarenavirus_choriomeningitidis|?|Homo_sapiens|France|?|S|LE|tile_16                                                                            572.169438
YP_006858706.1|precursor|Lunk_virus_NKS-1|?|Mus_minutoides|Zambia:Lusaka|10-Aug-2010|S|NKS-1|tile_17                                                                   559.508498
WDW20703.1|precursor|Ngerengere_mammarenavirus|TZ23187|Mus_minutoides_SE|Tanzania:_Mkundi|08-Dec-2009|S|?|tile_17                                                      491.390222
XOB76280.1|glycoprotein|Mammarenavirus_ryukyuense|CXMIX-YNSF117|Mus_musculus|China:_Yunnan|2021|S|?|tile_17                                                            468.577622
AFI42021.1|precursor|Mammarenavirus_choriomeningitidis|

{'pearsons_r': np.float64(0.9970396135453448),
 'top_peptides': ['KFWYLNNTKTGQVSLPQCWLISNGSYLNETHFSADIEREANNLITE',
  'TVNSLISDQLLMRNHLRDLMGVPYCNYSKFWYLEHAKTGETSVPKC',
  'YCNYSKFWYLEHAKTGETSVPKCWLVTNGSYLNETHFSDQIEQEAD',
  'TINSLISDQLLMRNHLRDLMGVPYCNYSKFWYLEHSKTNETSVPKC',
  'KFWYLNNTKTGQVSLPQCWLISNGSYLNKTDFTDDIEREANNLVTE',
  'SKFWYLNDTRNGKTSLPKCWLISNGSYLNETHFSDQIEQEADNMIT',
  'YSKFWYLEHYKTGETSLPKCWLVSNGSYLNETHFSTEIEQEADNMI',
  'YTKFWFLNDTRTGRTSLPKCWKVSNNTYLNETEFSTEIEQEADNMI',
  'LKELLDTPYCNYTKFWYVNHTITGEHSLPRCWMVKNNSYLNESEFR',
  'NYSKFWYLNDTRNGRTSLPKCWLVSNGSYLNETQFSDTIEQEADNM',
  'NDQLIMKNHLRDIMGIPYCNYSKYWYLNNTVTGKSSLPKCWLVSNG',
  'YSKFWYLEHYKTGETSLPKCWLVSNGSYLNETHFSDEIEQEANNMI',
  'KQLARIPYCNYTKFWYINDTVSGKHSLPQCWKVHNGSYLNESQFKN',
  'TRFWYLNNTKTGQVSLPKCWLISNGSYLNQTSFTDDIEREANNLIT',
  'TVTPLISDQLLMRNHLRDLMGVPYCNSSKFWYLEHAKTGETSVPKC',
  'TVNSLISDQLLMRNHLRDLMGVPYCNYSKFWYLEHVKTKETSVPKC',
  'YCNYSKFWYLKHVKTGETSVPKCWLISNGSYLNETHFSDQIEQEAD',
  'YCNYSKFWYLEHSRTGESSTPKCWLVSTGSYLNETHFSDQIEQEAD',
 

In [147]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='13-4E', literature_note = "Literature reports LASV 13-4E to bind a linear epitope within the T-loop of the GP2 subunit of GPC, mapped to residues aa361-375, non-neutralizing")

Pearson's r = 0.9983442255779336 for 13-4E mAbs
peptide
QIM73713.1|precursor|Arenavirus_sp.|RtRsp-AreV/Tt2018|Rattus_sp.|Thailand|Nov-2018|?|?|tile_17                                                            2717.902187
AZI96059.1|precursor|Mammarenavirus_lassaense|LASV/H.sapiens-tc/NGA/2012/IRR_006|Homo_sapiens|Nigeria:_Rivers|21-Feb-2012|S|?|tile_16                     1267.202069
XOB76283.1|precursor|Yunnan_rodent_mammarenavirus_1|HHJP-YNSC97|Niviventer_fulvescens|China:_Yunnan|2021|?|?|tile_17                                      1015.159901
QWQ58032.1|precursor|Bitu_virus|ANG0070|Micaelamys_namaquensis|Angola:_Tundavala|Jul-2017|S|?|tile_17                                                      780.398812
AZI96476.1|precursor|Mammarenavirus_lassaense|LASV/H.sapiens-wt/NGA/2018/IRR_119|Homo_sapiens|Nigeria:_EDO|25-Feb-2018|S|?|tile_16                         691.499348
QIM73713.1|precursor|Arenavirus_sp.|RtRsp-AreV/Tt2018|Rattus_sp.|Thailand|Nov-2018|?|?|tile_16                    

{'pearsons_r': np.float64(0.9983442255779336),
 'top_peptides': ['KFWYLNNTKTGQVSLPQCWLISNGSYLNETHFSADIEREANNLITE',
  'YCNYSKFWYLEHAKTGETSVPKCWLVTNGSYLNETHFSDQIEQEAD',
  'KELLGIPYCNYTKFWYLNNTSLNLHSLPKCWLVSNNSYLNKTEFSF',
  'NDQLIMKNHLRDIMGIPYCNYSRYWYLNHTTTGKTSLPKCWLVSNG',
  'INDQLIMRNHLKDLMGIPYCNYTKFWYLNNTKTGQVSLPQCWLISN',
  'KFWYLNNTKTGQVSLPQCWLISNGSYLNKTDFTDDIEREANNLVTE',
  'RFWYLNNTRTGQVSLPQCWLITNGSYLNKTEFTNDIEREANNLVTE',
  'SKFWYLNDTRNGKTSLPKCWLISNGSYLNETHFSDQIEQEADNMIT',
  'DQLIMKNHLRDIMGIPYCNYSRYWYLNHTSTGKTSLPRCWLISNGS',
  'NYSKFWYLNDTRNGRTSLPKCWLVSNGSYLNETQFSDTIEQEADNM',
  'NDQLIIKNHLRDLMGIPYCNYSKFWYLNDTRNGKTSLPKCWKVSNN',
  'LGIPYCNYTKFWYLNSTRTGHHSLPKCWMISNGSYLNETHFSTEIE',
  'DQLIMKNHLRDIMGIPYCNYSKYWYLNHTSSGRTSLPKCWLVSNGS',
  'DQLIMKNHLRDIMGIPYCNYSKYWYLNHTITGKTSLPRCWLVSNGS',
  'DQLIMKNHLRDIMGIPYCNYSKYWYLNHTVTGKTSLPKCWLVSNGS',
  'NDQLIMKNHLRDIMGIPYCNYSKYWYLNNTVTGKSSLPKCWLVSNG',
  'YCNYTKFWYLNSTKTGVVSLPKCWLVSNGSYLNVTHFSNDIEQEAD',
  'TRFWYLNNTKTGQVSLPKCWLISNGSYLNQTSFTDDIEREANNLIT',
 

In [ ]:
analyze_mab_peptides(df = df_mAb_fc_arenaviridae_gp, peps_metadata = peps_metadata, regex='13-4E', literature_note = "Literature reports LASV 13-4E to bind a linear epitope within the T-loop of the GP2 subunit of GPC, mapped to residues aa361-375, non-neutralizing")

In [124]:
mAb13_4e = df_mAb_fc_arenaviridae_gp.filter(regex='13-4E')

In [125]:
# test pearson's correlation

pearsons_r = mAb13_4e.iloc[:, 0].corr(mAb13_4e.iloc[:, 1], method='pearson')

print(f"Pearson's r = {pearsons_r:.3}")

Pearson's r = 0.998


In [127]:
mAb13_4e_avg = mAb13_4e.mean(axis=1)

mAb13_4e = mAb13_4e_avg.sort_values(ascending=False).head(20)

mAb13_4e_peptide_sequences = peps_metadata.loc[peps_metadata['seq_id'].isin(mAb13_4e.index.to_list())]

In [130]:
mAb13_4e = mAb13_4e.sort_values(ascending=False).head(20)
mAb13_4e_peptide_sequences = peps_metadata.loc[peps_metadata['seq_id'].isin(mAb13_4e.index.to_list())]

#### Aligning to Josiah LASV GPC refseq

In [131]:
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio import Align, SeqIO

ref_path = "../ref_seqs/UP000002473_11622_LASV_Josiah_reference.fasta"

#SeqIO.to_dict() expects parsed sequence records
ref_records = SeqIO.to_dict(SeqIO.parse(ref_path, 'fasta'))

target_id = "sp|P08669|GLYC_LASSJ"
ref_seq = ref_records[target_id].seq

#configure pairwise aliger for peptides 
aligner = Align.PairwiseAligner()
aligner.mode = 'local' #for peptides
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

#align each peptide to the reference sequence
peptides = mAb13_4e_peptide_sequences['sequence'].to_list()

positions_covered = []

for pep in peptides:
    alignments = aligner.align(ref_seq, pep)
    best_alignment = alignments[0] #top scoring alignment

    print(f"Peptide: {pep} | Score: {best_alignment.score}")

    #print start and end coordinated on the reference protein (0-based)
    ref_start, ref_end = best_alignment.aligned[0][0]
    print(f"Mapped to reference positions {ref_start + 1} to {ref_end}")
    print(best_alignment)

    for i in range(ref_start+1, ref_end):
        if i in positions_covered:
            continue
        else:
            positions_covered.append(i)

positions_covered = sorted(positions_covered)

Peptide: KFWYLNNTKTGQVSLPQCWLISNGSYLNETHFSADIEREANNLITE | Score: 204.0
Mapped to reference positions 368 to 413
target          367 KYWYLNHTTTGRTSLPKCWLVSNGSYLNETHFSDDIEQQADNMITE 413
                  0 |.||||.|.||..|||.|||.||||||||||||.|||..|.|.|||  46
query             0 KFWYLNNTKTGQVSLPQCWLISNGSYLNETHFSADIEREANNLITE  46

Peptide: YCNYSKFWYLEHAKTGETSVPKCWLVTNGSYLNETHFSDQIEQEAD | Score: 225.0
Mapped to reference positions 363 to 408
target          362 YCNYSKYWYLNHTTTGRTSLPKCWLVSNGSYLNETHFSDDIEQQAD 408
                  0 ||||||.|||.|..||.||.||||||.||||||||||||.|||.||  46
query             0 YCNYSKFWYLEHAKTGETSVPKCWLVTNGSYLNETHFSDQIEQEAD  46

Peptide: KELLGIPYCNYTKFWYLNNTSLNLHSLPKCWLVSNNSYLNKTEFSF | Score: 188.0
Mapped to reference positions 356 to 400
target          355 RDIMGIPYCNYSKYWYLNHTTTGRTSLPKCWLVSNGSYLNETHFS 400
                  0 ....|||||||.|.||||.|.....||||||||||.||||.|.||  45
query             0 KELLGIPYCNYTKFWYLNNTSLNLHSLPKCWLVSNNSYLNKTEFS  45

Peptide: NDQLIMKNHLRDIMGI